# AGender-MTL - Step 3: external check on Iranian Churn (CPU)

The age part of the pipeline on a second operator's data: Iranian Churn (UCI 563, CC BY 4.0), 3,150
customers aggregated over nine months, five age groups, no gender.

- `Age` holds one value per age group, so it is the label written as a number and is dropped.
- `Customer Value` is a linear formula of usage whose coefficients differ between age groups, so it also
  carries the label and is dropped (the check is saved).
- Exact copies of rows are removed before splitting, so no customer is in training and test at once.
- 70 / 15 / 15 split stratified by age group, seeds 42, 7, 123, 2024, 99; encoding as for LAT.
- QUBO relevance to the age group; the number of attributes kept (4, 6, 8, 10 or all 11) is chosen on the
  validation rows of each seed.
- Models: the network's age head (gender weight 0), Logistic Regression, Random Forest, Gradient Boosting;
  then the oversampling methods of Step 2 for the network and Gradient Boosting.

**Kaggle settings:** Accelerator = **None** (CPU). **Internet = On** (downloads the data from UCI and
installs `ctgan`). **Input:** `agender-mtl-code`. **Expected run time:** about 15-25 minutes.

In [ ]:
import os
import sys
from pathlib import Path

# The code is attached as the Kaggle dataset `agender-mtl-code`. Find the `agender_mtl` package
# wherever Kaggle placed it, and make it importable.
def find_file(name, root='/kaggle/input'):
    for folder, _, files in os.walk(root, followlinks=True):
        if name in files:
            return Path(folder) / name
    return None


package = find_file('benchmark.py')
if package is None:
    attached = sorted(str(p) for p in Path('/kaggle/input').glob('*/*'))[:20]
    raise RuntimeError('The agender-mtl-code dataset is not attached to this notebook. In the editor: '
                       'Add Input -> Your Datasets -> agender-mtl-code, then Save & Run All again. '
                       f'Attached now: {attached or "nothing"}')
if package.parent.name == 'agender_mtl' and (package.parent / '__init__.py').exists():
    code_root = package.parent.parent
else:
    # Kaggle sometimes flattens an uploaded folder; link the files under the name `agender_mtl`.
    code_root = Path('/kaggle/working/code')
    code_root.mkdir(parents=True, exist_ok=True)
    if not (code_root / 'agender_mtl').exists():
        os.symlink(package.parent, code_root / 'agender_mtl')

sys.path.insert(0, str(code_root))
import agender_mtl
print('Code loaded from', Path(agender_mtl.__file__).parent)

In [ ]:
# ---- Settings ----
OUTPUT = Path('/kaggle/working/agender_mtl_outputs/step3')
TIME_BUDGET_MINUTES = 120

In [ ]:
import importlib.util
import json
import subprocess

for package, module in (('imbalanced-learn', 'imblearn'), ('ctgan', 'ctgan')):
    if importlib.util.find_spec(module) is None:
        done = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', package], capture_output=True, text=True)
        print(package, 'installed' if done.returncode == 0 else 'could not be installed: ' + done.stderr[-500:])

from agender_mtl import churn, data, environment, step3
from agender_mtl.runner import StepRunner

methods = step3.METHODS if importlib.util.find_spec('ctgan') else [m for m in step3.METHODS if m != 'CTGAN']
runner = StepRunner(OUTPUT, TIME_BUDGET_MINUTES)
summary = {'methods': methods}
summary['environment'] = runner.run('measure the machine', environment.collect)
summary['download'] = runner.run('download Iranian Churn', churn.fetch, OUTPUT / 'data')
raw = runner.run('read', churn.read_raw, OUTPUT / 'data' / 'iranian_churn.zip')
table = churn.clean(raw)
summary['data'] = churn.describe(raw, table)
summary['customer_value_formula'] = runner.run('customer value formula check', churn.formula_check, raw)
summary['near_copies'] = runner.run('near-copy check', churn.near_copy_check, table)
print(json.dumps(summary['data'], indent=1, default=str))

In [ ]:
summary['result'] = runner.run('age benchmark and oversampling, five seeds', step3.run, table, OUTPUT, methods=methods)
for name, scores in (summary['result'] or {}).get('test_means', {}).items():
    print(f'{name:45s}', scores)

summary['steps'] = runner.steps
with open(OUTPUT / 'step3_summary.json', 'w') as f:
    json.dump(summary, f, indent=1, default=str)
print('Failed or skipped steps:', runner.failed() or 'none')